# Export to Ollama (GGUF)

Turns the final **SFT + DPO** model (`outputs/dpo_qwen2.5-0.5b/merged/`) into a **GGUF** file (one file: weights + tokenizer + config) and registers it in Ollama, so it runs with `ollama run` on CPU or GPU without Python.

**Tags:** the DPO model is published as **`:dpo`** and **`:latest`**; the earlier SFT model is kept as **`:sft`** (pushing without a tag would overwrite `:latest` and the SFT version would be gone).

**Steps:** 1. config → 2. get llama.cpp's converter → 3. convert to GGUF → 4. write the Modelfile → 5. `ollama create` → 6. test → 7. accuracy check → 8. publish to ollama.com (optional).

**Needs:** Ollama installed and running, and `git`.

## 1. Config
- **`QUANT = "q8_0"`**: 8-bit, ~530 MB, almost no accuracy loss. (`"bf16"` = full precision, ~950 MB.)
- **`OLLAMA_NAME`**: the name you'll use in `ollama run`. **`TAG = "dpo"`** is this model's version tag.

In [ ]:
import json, sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(REPO_ROOT))
NOTEBOOKS = REPO_ROOT / "notebooks"

RUN_NAME = "dpo_qwen2.5-0.5b"            # final model: SFT + DPO (merged)
MERGED_DIR = NOTEBOOKS / "outputs" / RUN_NAME / "merged"
RESULTS_DIR = NOTEBOOKS / "data" / "eval" / "results" / RUN_NAME

QUANT = "q8_0"
OLLAMA_NAME = "qwen2.5_0.5b-instruct-fc-Chat"
TAG = "dpo"                               # this version; also copied to :latest

GGUF_DIR = NOTEBOOKS / "outputs" / RUN_NAME / "gguf"           # gitignored
GGUF_FILE = GGUF_DIR / f"{OLLAMA_NAME}-{QUANT}.gguf"
LLAMA_CPP = REPO_ROOT / "third_party" / "llama.cpp"             # gitignored
GGUF_DIR.mkdir(parents=True, exist_ok=True)
print("Model folder:", MERGED_DIR, "| exists:", MERGED_DIR.exists())

## 2. Get llama.cpp's converter
We only need its Python script `convert_hf_to_gguf.py`; no compiling. It uses the `gguf` library bundled in the same repo, and `torch`/`transformers`, which your env already has.

⚠️ Don't `pip install -r` llama.cpp's requirements file: it can replace your CUDA PyTorch with a CPU-only build. If the script complains about one missing package, install just that one.

In [7]:
import subprocess

if not LLAMA_CPP.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(LLAMA_CPP)], check=True)
print("Converter:", (LLAMA_CPP / "convert_hf_to_gguf.py").exists())

Converter: True


## 3. Convert to GGUF
Reads the safetensors + config + tokenizer and writes one `.gguf` file. With `--outtype q8_0` it converts and quantizes in one go. Takes about a minute.

In [8]:
subprocess.run([sys.executable, str(LLAMA_CPP / "convert_hf_to_gguf.py"), str(MERGED_DIR),
                "--outfile", str(GGUF_FILE), "--outtype", QUANT], check=True)
print(f"\n✅ {GGUF_FILE.name}: {GGUF_FILE.stat().st_size / 1024**2:.0f} MB "
      f"(vs {(MERGED_DIR / 'model.safetensors').stat().st_size / 1024**2:.0f} MB bf16)")


✅ qwen2.5_0.5b-instruct-fc-Chat-q8_0.gguf: 506 MB (vs 942 MB bf16)


## 4. Write the Modelfile
A **Modelfile** is Ollama's recipe:
- **`FROM`**: the GGUF file.
- **`TEMPLATE`**: Qwen's chat format (`<|im_start|>role ... <|im_end|>`), same as training. Like Qwen's own template, it adds
  Qwen's default system line when the caller sends no system message.
- **No `SYSTEM` prompt baked in** (on purpose, same as the Hugging Face upload). The *caller* sends the tool-calling system prompt
  (`build_messages` does it, and so does the gateway). Baking it in forces every chat into tool mode: plain "hi" in a chat app
  then comes back as an invented call like `hello(name="world")`.
- **`PARAMETER`**: `temperature 0` (greedy, like the eval), **`repeat_penalty 1.0`** (Ollama's default 1.1 punishes the
  `"`, `{`, `:` that JSON repeats constantly), context 4096, stop at `<|im_end|>`.


In [9]:
TEMPLATE = """{{- if not (and .Messages (eq (index .Messages 0).Role "system")) }}<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
{{ end }}{{- range .Messages }}<|im_start|>{{ .Role }}
{{ .Content }}<|im_end|>
{{ end }}<|im_start|>assistant
"""

modelfile = f'''FROM ./{GGUF_FILE.name}

TEMPLATE """{TEMPLATE}"""

PARAMETER temperature 0
PARAMETER repeat_penalty 1.0
PARAMETER num_ctx 4096
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
'''
(GGUF_DIR / "Modelfile").write_text(modelfile, encoding="utf-8")
print(modelfile)


FROM ./qwen2.5_0.5b-instruct-fc-Chat-q8_0.gguf

TEMPLATE """{{- if not (and .Messages (eq (index .Messages 0).Role "system")) }}<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
{{ end }}{{- range .Messages }}<|im_start|>{{ .Role }}
{{ .Content }}<|im_end|>
{{ end }}<|im_start|>assistant
"""

PARAMETER temperature 0
PARAMETER repeat_penalty 1.0
PARAMETER num_ctx 4096
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"



## 5. Register it in Ollama
1. **Keep the SFT version first:** your local `qwen2.5_0.5b-instruct-fc-Chat` (= `:latest`) is still the SFT model. It's copied to **`:sft`** before anything overwrites it (only once: skipped if `:sft` already exists).
2. `ollama create` registers the DPO GGUF as **`:dpo`**, then it's copied to **`:latest`**, so plain `ollama run qwen2.5_0.5b-instruct-fc-Chat` gives the new model.

Copies are cheap: Ollama stores each weight file once and tags just point at it.

In [ ]:
listed = subprocess.run(["ollama", "list"], capture_output=True, text=True, check=True).stdout
if f"{OLLAMA_NAME}:sft" not in listed and f"{OLLAMA_NAME}:latest" in listed:
    subprocess.run(["ollama", "cp", OLLAMA_NAME, f"{OLLAMA_NAME}:sft"], check=True)
    print(f"kept the SFT model as {OLLAMA_NAME}:sft")

subprocess.run(["ollama", "create", f"{OLLAMA_NAME}:{TAG}", "-f", "Modelfile"], cwd=GGUF_DIR, check=True)
subprocess.run(["ollama", "cp", f"{OLLAMA_NAME}:{TAG}", f"{OLLAMA_NAME}:latest"], check=True)
subprocess.run(["ollama", "list"], check=True)

## 6. Test it: tool request vs plain chat
- **Tool request:** send the full trained prompt with `build_messages` (system prompt + tools + request). Expect JSON.
- **No tool fits:** "Tell me a joke" with only a weather tool. Expect `[]` (DPO taught this; SFT called `get_weather`).
- **Plain chat:** "What is an LLM?" with **no** system prompt. Expect a normal text answer. This only works because no tool prompt is baked into the Modelfile.


In [ ]:
import urllib.request

from src.prompts import build_messages


def ollama_chat(messages):
    body = json.dumps({"model": OLLAMA_NAME, "messages": messages, "stream": False,
                       "options": {"temperature": 0, "repeat_penalty": 1.0}}).encode()
    req = urllib.request.Request("http://localhost:11434/api/chat", data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read())["message"]["content"]


record = {
    "query": "What's the weather like in Pune right now?",
    "tools": [{"name": "get_weather", "description": "Current weather for a city.",
               "parameters": {"city": {"type": "str", "description": "City name"}}}],
}
print("WITH TOOLS :", ollama_chat(build_messages(record)))
print("NO TOOL FITS:", ollama_chat(build_messages({**record, "query": "Tell me a joke"})))
print("PLAIN CHAT :", ollama_chat([{"role": "user", "content": "What is an LLM? One sentence."}])[:300])

## 7. Accuracy check: did GGUF/Q8 cost anything?
Run 100 questions from the frozen test set through Ollama, score them with **the same scorer**, and compare with the bf16 model's scores on the **same 100 questions**. A drop of ~1 point or less is normal for Q8_0.

In [13]:
import json
import time
import urllib.request

import pandas as pd

from src.data_utils import read_jsonl
from src.evaluate import score_prediction
from src.prompts import build_messages


def ollama_chat(messages, timeout=300):
    body = json.dumps({
        "model": OLLAMA_NAME,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": 0,
            "repeat_penalty": 1.0,
            "num_predict": 512,   # hard cap: a looping answer stops at 512 tokens instead of running forever
        },
    }).encode()
    req = urllib.request.Request("http://localhost:11434/api/chat", data=body,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read())["message"]["content"]


N = 100
test = read_jsonl(NOTEBOOKS / "data" / "eval" / "test_v1.jsonl")[:N]

gguf_em, slow, failed = [], [], []
t0 = time.time()
for i, r in enumerate(test):
    t = time.time()
    try:
        reply = ollama_chat(build_messages(r))
    except Exception as e:                      # one bad request shouldn't kill the whole run
        failed.append((r["id"], str(e)))
        reply = ""
    gguf_em.append(score_prediction(reply, r)["exact_match"])
    if time.time() - t > 20:                    # normal answers take ~1s
        slow.append((r["id"], round(time.time() - t), reply[-120:]))
    if (i + 1) % 20 == 0:
        print(f"{i + 1}/{N} done, {time.time() - t0:.0f}s")

bf16 = pd.read_csv(RESULTS_DIR / "scored.csv").set_index("id").loc[[r["id"] for r in test], "exact_match"]
print(f"\nExact match on {N} questions:  bf16 {100 * bf16.mean():.1f}%  |  GGUF {QUANT} {100 * sum(gguf_em) / N:.1f}%")
print("Slow answers (>20s):", slow or "none")
print("Failed requests:", failed or "none")
print(f"Run it yourself:  ollama run {OLLAMA_NAME}")

20/100 done, 50s
40/100 done, 102s
60/100 done, 152s
80/100 done, 204s
100/100 done, 255s

Exact match on 100 questions:  bf16 79.0%  |  GGUF q8_0 81.0%
Slow answers (>20s): none
Failed requests: none
Run it yourself:  ollama run qwen2.5_0.5b-instruct-fc-Chat


## 8. Publish to ollama.com (optional)
Steps 5–7 only registered the model **on your PC**. To let anyone run `ollama run <you>/<model>`:

1. Make an account on **ollama.com** (your username becomes the namespace).
2. **Add this PC's public key** (one time): ollama.com → Settings → **Ollama keys** → Add, and paste the key printed below.
   Ollama created this key pair when it was installed. The private half (`id_ed25519`) never leaves your PC, and pushes are signed with it,
   so there's no token in the code. Same idea as SSH keys on GitHub.
3. Run the cell: it pushes three tags to `<username>/<name>`: **`:dpo`** and **`:latest`** (this model) and **`:sft`** (the earlier model, if kept in step 5). The weights upload once (~530 MB per distinct model); extra tags are almost free.

Update the description on ollama.com: say `:latest`/`:dpo` = SFT + DPO (answers `[]` when no tool fits), `:sft` = SFT only, and link the Hugging Face model card for the prompt format and results.


In [14]:
OLLAMA_USER = "Avahan"
PUBLIC_NAME = f"{OLLAMA_USER}/qwen2.5_0.5b-instruct-fc-Chat"   # must match the page you created on ollama.com

pub_key = Path.home() / ".ollama" / "id_ed25519.pub"
print("Paste this into ollama.com -> Settings -> Ollama keys:\n")
print(pub_key.read_text().strip() if pub_key.exists() else f"Not found at {pub_key}, has Ollama been run once on this PC?")


Paste this into ollama.com -> Settings -> Ollama keys:

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIIu5NByerx+vPqCd5O6iASdfGk2EQiaMjn6rJmtJeFHB


In [ ]:
# run after the key is added on ollama.com
listed = subprocess.run(["ollama", "list"], capture_output=True, text=True, check=True).stdout
tags = [t for t in ("sft", TAG, "latest") if f"{OLLAMA_NAME}:{t}" in listed]   # sft first, latest last
for t in tags:
    subprocess.run(["ollama", "cp", f"{OLLAMA_NAME}:{t}", f"{PUBLIC_NAME}:{t}"], check=True)   # namespaced copy
    subprocess.run(["ollama", "push", f"{PUBLIC_NAME}:{t}"], check=True)
    print(f"pushed {PUBLIC_NAME}:{t}")
print(f"✅ Published: https://ollama.com/{PUBLIC_NAME}/tags")
print(f"   ollama run {PUBLIC_NAME}        (= :{TAG}, SFT + DPO)")
print(f"   ollama run {PUBLIC_NAME}:sft    (earlier SFT-only model)")
